In [0]:
# Databricks notebook source
# src/pipelines/gold_pipeline.py

import dlt
from pyspark.sql import functions as F

# =========================================================================
# 1. GOLD CONFORMED DIMENSIONS
# =========================================================================

@dlt.table(
    name="gold.dim_users",
    comment="Conformed Users SCD Type 2 dimension. Grain: one row per user version."
)
def gold_dim_users():
    return (
        dlt.read("silver.dim_users")
           .select(
               F.col("user_id").alias("user_id"),
               F.col("name").alias("user_name"),
               F.col("email").alias("email"),
               F.col("country").alias("country"),
               F.col("user_type").alias("user_type"),
               F.col("is_business").alias("is_business"),
               F.col("company_name").alias("company_name"),
               F.col("__start_at").alias("valid_from"),
               F.col("__end_at").alias("valid_to")
           )
    )

@dlt.table(
    name="gold.dim_properties",
    comment="Conformed Properties dimension. Grain: one row per property listing."
)
def gold_dim_properties():
    return (
        dlt.read("silver.dim_properties")
           .select(
               "property_id",
               "host_id",
               "destination_id",
               "title",
               "description",
               "base_price",
               "property_type",
               "max_guests",
               "bedrooms",
               "bathrooms",
               "property_latitude",
               "property_longitude"
           )
    )

# =========================================================================
# 2. GOLD FACT SESSIONS (Session Aggregation + Attribution + PIT Join)
# =========================================================================

@dlt.table(
    name="gold.fct_sessions",
    comment="Fact table of user browsing sessions. Grain: one row per user session."
)
def gold_fct_sessions():
    clicks = dlt.read("silver.silver_clickstream")
    bookings = dlt.read("silver.silver_bookings")
    users = dlt.read("silver.dim_users")

    # Step 2A: Aggregate clickstream events to the session grain (30-min window)
    session_agg = (
        clicks.groupBy("session_id", "user_id")
              .agg(
                  F.min("event_timestamp").alias("session_start_time"),
                  F.max("event_timestamp").alias("session_end_time"),
                  F.count("event").alias("event_count"),
                  F.first("device_type", ignorenulls=True).alias("device_type"),
                  F.first("referrer", ignorenulls=True).alias("referrer"),
                  F.first("property_id", ignorenulls=True).alias("property_id"),
                  F.max(F.when(F.col("event") == "search", 1).otherwise(0)).alias("has_search"),
                  F.max(F.when(F.col("event") == "view", 1).otherwise(0)).alias("has_view"),
                  F.max(F.when(F.col("event").isin("booking_attempt", "confirm_booking"), 1).otherwise(0)).alias("has_booking_attempt")
              )
    )

    # Step 2B: Session-to-Booking Attribution
    # Attributed if booking created between session start and session end + 30 min window
    booking_join = (
        session_agg.join(
            bookings.select(
                F.col("booking_id"),
                F.col("user_id").alias("b_user_id"),
                F.col("created_at").alias("booking_time")
            ),
            (session_agg.user_id == F.col("b_user_id")) & 
            (F.col("booking_time") >= session_agg.session_start_time) & 
            (F.col("booking_time") <= session_agg.session_end_time + F.expr("INTERVAL 30 MINUTES")),
            "left"
        ).drop("b_user_id")
    )

    with_attribution = (
        booking_join.withColumn("is_booked", F.when(F.col("booking_id").isNotNull(), 1).otherwise(0))
    )

    # Step 2C: Point-in-Time Join to Users SCD Type 2
    return (
        with_attribution.join(
            users,
            (with_attribution.user_id == users.user_id) &
            (with_attribution.session_start_time >= users.__start_at) &
            ((with_attribution.session_start_time < users.__end_at) | users.__end_at.isNull()),
            "left"
        ).select(
            with_attribution["session_id"],
            with_attribution["user_id"],
            F.coalesce(users["country"], F.lit("Unknown")).alias("user_country"),
            F.coalesce(users["user_type"], F.lit("standard")).alias("user_type"),
            with_attribution["property_id"],
            with_attribution["booking_id"],
            with_attribution["device_type"],
            with_attribution["referrer"],
            with_attribution["session_start_time"],
            with_attribution["session_end_time"],
            with_attribution["event_count"],
            with_attribution["has_search"],
            with_attribution["has_view"],
            with_attribution["has_booking_attempt"],
            with_attribution["is_booked"]
        )
    )

# =========================================================================
# 3. GOLD FACT BOOKINGS
# =========================================================================

@dlt.table(
    name="gold.fct_bookings",
    comment="Fact table of completed bookings. Grain: one row per booking transaction."
)
def gold_fct_bookings():
    bookings = dlt.read("silver.silver_bookings")
    users = dlt.read("silver.dim_users")

    return (
        bookings.join(
            users,
            (bookings.user_id == users.user_id) &
            (bookings.created_at >= users.__start_at) &
            ((bookings.created_at < users.__end_at) | users.__end_at.isNull()),
            "left"
        ).select(
            bookings["booking_id"],
            bookings["user_id"],
            F.coalesce(users["country"], F.lit("Unknown")).alias("user_country"),
            bookings["property_id"],
            bookings["check_in"],
            bookings["check_out"],
            bookings["guests_count"],
            bookings["total_amount"],
            bookings["booking_status"],
            bookings["created_at"].alias("booking_timestamp")
        )
    )

# =========================================================================
# 4. GOLD FACT REVIEWS (Soft-Delete Indicator Preserved)
# =========================================================================

@dlt.table(
    name="gold.fct_reviews",
    comment="Fact table of property reviews and ratings. Grain: one row per submitted review."
)
def gold_fct_reviews():
    reviews = dlt.read("silver.silver_reviews")
    users = dlt.read("silver.dim_users")

    return (
        reviews.join(
            users,
            (reviews.user_id == users.user_id) &
            (reviews.created_at >= users.__start_at) &
            ((reviews.created_at < users.__end_at) | users.__end_at.isNull()),
            "left"
        ).select(
            reviews["review_id"],
            reviews["booking_id"],
            reviews["user_id"],
            F.coalesce(users["country"], F.lit("Unknown")).alias("user_country"),
            reviews["property_id"],
            reviews["rating"],
            reviews["review_comment"],
            reviews["is_deleted"],
            reviews["created_at"].alias("review_timestamp")
        )
    )